## Unified API


In [ ]:
import logging
import litellm
from litellm import completion

# Hide LiteLLM fallback errors, warnings, and debug logs
litellm.set_verbose = False
litellm.suppress_debug_info = True
litellm.disable_streaming_logging = True

logging.getLogger("LiteLLM").setLevel(logging.CRITICAL)
logging.getLogger("litellm").setLevel(logging.CRITICAL)

In [77]:
import os
from dotenv import load_dotenv
load_dotenv()

print("OpenAI key loaded:", "✅" if os.getenv("OPENAI_API_KEY") else "❌")
print("Groq key loaded:", "✅" if os.getenv("GROQ_API_KEY") else "❌")
print("Gemini key loaded:", "✅" if os.getenv("GEMINI_API_KEY") else "❌")
print("Mistral key loaded:", "✅" if os.getenv("MISTRAL_API_KEY") else "❌")

OpenAI key loaded: ❌
Groq key loaded: ✅
Gemini key loaded: ❌
Mistral key loaded: ✅


In [78]:
# response_groq = completion(
#     model="groq/llama-3.3-70b-versatile",
#     messages=[{"role": "user", "content": "Explain RAG in one sentence."}]
# )
# print("🟢 Groq:", response_groq.choices[0].message.content)

response_ollama = completion(
    model="ollama/llama3.2:latest",
    messages=[{"role": "user",
               "content": "Explain Retrieval-Augmented Generation in one sentence."}]
)
print("🔵 Ollama:", response_ollama.choices[0].message.content)


# response_mistral = completion(
#     model="mistral/mistral-small-latest",
#     messages=[{"role": "user",
#                "content": "Explain Retrieval-Augmented Generation in one sentence."}]
# )
# print("🔴 Mistral:", response_mistral.choices[0].message.content)

🔵 Ollama: Retrieval-Augmented Generation (RAG) is a machine learning technique that combines the strengths of retrieval-based models with generation-based models to generate high-quality text by retrieving relevant information from a database and then using this retrieved information as input to guide the generation process.


## Automatic Fallbacks


In [79]:
# Define a fallback chain: try Gemini first -> Openai -> Ollama -> Groq -> Mistral
response = completion(
    model="gemini/gemini-1.5-flash",
    messages=[{"role": "user",
               "content": "Explain Retrieval-Augmented Generation in one sentence."}],
    fallbacks=[
        "openai/gpt-4o-mini",
        "ollama/llama3.2:latest",
        "groq/qwen/qwen3-32b",
        "mistral/mistral-small-latest"
    ]
)

print("\n✅ Response:", response.choices[0].message.content)
print("🤖 Model used:", response.model)

Task was destroyed but it is pending!
task: <Task pending name='Task-289' coro=<LoggingWorker._worker_loop() running at c:\Users\yasho\OneDrive\Desktop\Agent_Evaluation\.venv\Lib\site-packages\litellm\litellm_core_utils\logging_worker.py:109>>



✅ Response: Retrieval-Augmented Generation (RAG) is a machine learning approach that combines the strengths of retrieval algorithms and generative models to retrieve relevant information from a large database or knowledge graph and then uses it as input for generating new text, such as articles or summaries.
🤖 Model used: ollama/llama3.2:latest


## Cost Tracking

LiteLLM automatically calculates the cost of every call using its built-in pricing database. No more surprise bills.


In [80]:
from litellm import completion_cost

# response = completion(
#     model="mistral/mistral-small-latest",
#     messages=[{"role": "user", "content": "Write a haiku about AI."}]
# )

# # Get the exact USD cost of this single call
# cost = completion_cost(completion_response=response)

# print("Response:    ", response.choices[0].message.content)
# print("\nInput tokens: ", response.usage.prompt_tokens)
# print("Output tokens:", response.usage.completion_tokens)
# print(f"Cost:         ${cost:.8f}")

In [81]:
response = completion(
    model="ollama/llama3.2:latest",
    messages=[{"role": "user", "content": "Write a haiku about AI."}]
)

# Get the exact USD cost of this single call
cost = completion_cost(completion_response=response)

print("Response:    ", response.choices[0].message.content)
print("\nInput tokens: ", response.usage.prompt_tokens)
print("Output tokens:", response.usage.completion_tokens)
print(f"Cost:         ${cost:.8f}")

Response:     Metal minds awake
Learning, growing, and forgetting
Artificial dawn

Input tokens:  35
Output tokens: 15
Cost:         $0.00000000


## Caching

If 100 users ask "What is RAG?", you don't need to call the LLM 100 times.


In [82]:
import litellm

# 🧹 Reset any callbacks/strategies left over from earlier cells
litellm.callbacks = []
litellm.success_callback = []
litellm.failure_callback = []
litellm._async_success_callback = []
litellm._async_failure_callback = []

# Also clear any router-strategy state
litellm.cache = None

print("✅ LiteLLM state reset")

✅ LiteLLM state reset


In [83]:
import time
from litellm.caching import Cache

# Enable in-memory caching (you can also use Redis in production)
litellm.cache = Cache(type="local")

prompt = "What does LLM stand for? Answer in one line."

In [84]:
# First call
start = time.time()
r1 = completion(
    model="ollama/llama3.2:latest",
    messages=[{"role": "user", "content": prompt}],
    caching=True
)
t1 = time.time() - start
print(f"❄️  First call (API): {t1:.2f}s — {r1.choices[0].message.content}")

❄️  First call (API): 0.73s — LLM stands for Large Language Model.


In [85]:
# Second call — served from cache, near-instant
start = time.time()
r2 = completion(
    model="ollama/llama3.2:latest",
    messages=[{"role": "user", "content": prompt}],
    caching=True
)
t2 = time.time() - start
print(f"⚡ Second call (cache): {t2:.4f}s — {r2.choices[0].message.content}")

print(f"\n🚀 Speedup: {t1/t2:.1f}x faster, and ZERO cost on the second call!")

⚡ Second call (cache): 0.0010s — LLM stands for Large Language Model.

🚀 Speedup: 696.6x faster, and ZERO cost on the second call!


## Smart Routing — The Right Model for the Right Job


In [86]:
from litellm import Router

model_list = [
    {
        "model_name": "fast-cheap",
        "litellm_params": {
            "model": "groq/llama-3.3-70b-versatile",
            "api_key": os.getenv("GROQ_API_KEY")
        }
    },
    {
        "model_name": "smart-coding",
        "litellm_params": {
            "model": "mistral/mistral-small-latest",
            "api_key": os.getenv("MISTRAL_API_KEY")
        }
    },
    {
        "model_name": "balanced",
        "litellm_params": {
            "model": "ollama/llama3.2:latest",
        }
    }
]

router = Router(model_list=model_list)

In [87]:
fast_response = router.completion(
    model="fast-cheap",
    messages=[{"role": "user", "content": "Summarize: AI is changing software."}]
)

code_response = router.completion(
    model="smart-coding",
    messages=[
        {"role": "user", "content": "Write a Python function to reverse a string."}]
)

general_response = router.completion(
    model="balanced",
    messages=[{"role": "user", "content": "What is the capital of France?"}]
)

print("⚡ Fast/cheap (Groq): ", fast_response.choices[0].message.content[:200])

print("\n🧠 Smart/coding (Mistral):\n",
      code_response.choices[0].message.content[:200])

print("\n⚖️ Balanced (Ollama):\n",
      general_response.choices[0].message.content[:200])

⚡ Fast/cheap (Groq):  Artificial Intelligence (AI) is revolutionizing the software industry in several ways:

1. **Automation**: AI-powered tools can automate repetitive and mundane tasks, freeing up developers to focus on

🧠 Smart/coding (Mistral):
 Here's a Python function to reverse a string:

```python
def reverse_string(s):
    return s[::-1]
```

### Explanation:
- The slice notation `[::-1]` is used to reverse the string.
  - The first `:` 

⚖️ Balanced (Ollama):
 The capital of France is Paris.
